In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
fraud = pd.read_csv(
    "../data/raw/Fraud_Data.csv"
)

ip_country = pd.read_csv(
    "../data/raw/IpAddress_to_Country.csv"
)

credit = pd.read_csv(
    "../data/raw/creditcard.csv"
)

In [ ]:
fraud["signup_time"] = pd.to_datetime(
    fraud["signup_time"]
)

fraud["purchase_time"] = pd.to_datetime(
    fraud["purchase_time"]
)

In [ ]:
fraud["ip_address"] = fraud["ip_address"].astype(np.int64)

ip_country["lower_bound_ip_address"] = (
    ip_country["lower_bound_ip_address"]
    .astype(np.int64)
)

ip_country["upper_bound_ip_address"] = (
    ip_country["upper_bound_ip_address"]
    .astype(np.int64)
)

In [ ]:
fraud = fraud.sort_values(
    "ip_address"
)

ip_country = ip_country.sort_values(
    "lower_bound_ip_address"
)

In [ ]:
fraud = pd.merge_asof(
    fraud,
    ip_country,
    left_on="ip_address",
    right_on="lower_bound_ip_address",
    direction="backward"
)

In [ ]:
fraud = fraud[
    fraud["ip_address"]
    <= fraud["upper_bound_ip_address"]
]

In [ ]:
fraud["country"].head()

In [ ]:
fraud["country"].isnull().sum()

In [ ]:
country_fraud = (
    fraud.groupby("country")["class"]
    .mean()
    .sort_values(ascending=False)
)

country_fraud.head(20)

In [ ]:
country_fraud.head(15).plot(
    kind="bar",
    figsize=(12,5)
)

plt.title(
    "Highest Fraud Rate Countries"
)

plt.show()

Several countries exhibit elevated fraud rates.

Country-level information appears to provide useful predictive signal and will be retained.

In [ ]:
fraud["time_since_signup"] = (
    fraud["purchase_time"]
    - fraud["signup_time"]
).dt.total_seconds()

In [ ]:
fraud["time_since_signup"].describe()

In [ ]:
fraud["hour_of_day"] = (
    fraud["purchase_time"].dt.hour
)

In [ ]:
fraud["day_of_week"] = (
    fraud["purchase_time"].dt.dayofweek
)

In [ ]:
fraud["is_weekend"] = (
    fraud["day_of_week"] >= 5
).astype(int)

In [ ]:
user_tx_count = (
    fraud.groupby("user_id")
    .size()
    .reset_index(
        name="user_transaction_count"
    )
)

In [ ]:
fraud = fraud.merge(
    user_tx_count,
    on="user_id",
    how="left"
)

In [ ]:
device_tx_count = (
    fraud.groupby("device_id")
    .size()
    .reset_index(
        name="device_transaction_count"
    )
)

In [ ]:
fraud = fraud.merge(
    device_tx_count,
    on="device_id",
    how="left"
)

In [ ]:
device_user_count = (
    fraud.groupby("device_id")
    ["user_id"]
    .nunique()
    .reset_index(
        name="users_per_device"
    )
)

In [ ]:
fraud = fraud.merge(
    device_user_count,
    on="device_id",
    how="left"
)

In [ ]:
fraud = fraud.sort_values(
    ["user_id","purchase_time"]
)

In [ ]:
fraud["previous_purchase_time"] = (
    fraud.groupby("user_id")
    ["purchase_time"]
    .shift()
)

In [ ]:
fraud["seconds_since_previous_transaction"] = (
    fraud["purchase_time"]
    - fraud["previous_purchase_time"]
).dt.total_seconds()

In [ ]:
fraud[
    "seconds_since_previous_transaction"
] = fraud[
    "seconds_since_previous_transaction"
].fillna(
    fraud[
        "seconds_since_previous_transaction"
    ].median()
)

In [ ]:
columns_to_drop = [

    "signup_time",

    "purchase_time",

    "device_id",

    "ip_address",

    "previous_purchase_time",

    "lower_bound_ip_address",

    "upper_bound_ip_address"

]

In [ ]:
fraud.drop(
    columns=columns_to_drop,
    inplace=True
)

In [ ]:
fraud.head()

In [ ]:
categorical_columns = [

    "source",

    "browser",

    "sex",

    "country"

]

In [ ]:
fraud = pd.get_dummies(
    fraud,
    columns=categorical_columns,
    drop_first=True
)

In [ ]:
numeric_columns = [

    "purchase_value",

    "age",

    "time_since_signup",

    "hour_of_day",

    "day_of_week",

    "user_transaction_count",

    "device_transaction_count",

    "users_per_device",

    "seconds_since_previous_transaction"

]

In [ ]:
fraud.to_csv(
    "../data/processed/fraud_processed.csv",
    index=False
)

In [ ]:
credit.head()

In [ ]:
scaler = StandardScaler()

credit["Amount"] = (
    scaler.fit_transform(
        credit[["Amount"]]
    )
)

In [ ]:
credit["Time"] = (
    scaler.fit_transform(
        credit[["Time"]]
    )
)

In [ ]:
credit.to_csv(
    "../data/processed/creditcard_processed.csv",
    index=False
)

In [ ]:
fraud.shape

In [ ]:
credit.shape